# 01 — Non-IID Data Distribution

This notebook:
1. Loads the raw diabetes dataset
2. Creates realistic non-IID splits across 3 simulated healthcare organisations
3. Validates the splits (no overlap, complete coverage)
4. Visualises the data heterogeneity
5. Saves per-client CSV files to `data/federated/`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import N_CLIENTS, DIRICHLET_ALPHA, TARGET_COLUMN, FEATURE_COLUMNS
from src.data_utils import (
    load_dataset,
    create_noniid_splits,
    save_client_data,
    compute_client_statistics,
)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

## 1. Load Dataset

In [ ]:
df = load_dataset()
print(f'Total records: {len(df):,}')
print(f'Features: {len(FEATURE_COLUMNS)}')
print(f'\nClass distribution:')
print(df[TARGET_COLUMN].value_counts(normalize=True).rename({0: 'No Diabetes', 1: 'Diabetes'}))

## 2. Create Non-IID Splits (Dirichlet)

In [ ]:
# Experiment with different alpha values
alphas = [0.1, 0.5, 1.0]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, alpha in zip(axes, alphas):
    splits = create_noniid_splits(df, n_clients=N_CLIENTS, alpha=alpha)
    rates = [splits[i][TARGET_COLUMN].mean() for i in range(N_CLIENTS)]
    ax.bar([f'Client {i}' for i in range(N_CLIENTS)], rates, color=sns.color_palette('Set2')[:N_CLIENTS])
    ax.set_title(f'Dirichlet α = {alpha}')
    ax.set_ylabel('Positive Rate')
    ax.set_ylim(0, 1)
    ax.axhline(df[TARGET_COLUMN].mean(), linestyle='--', color='black', linewidth=1, label='Global rate')
    ax.legend(fontsize=8)

plt.suptitle('Non-IID Class Distribution Across Clients', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/data_distribution_alpha_comparison.png', bbox_inches='tight')
plt.show()

## 3. Create Final Splits (configured alpha)

In [ ]:
client_dfs = create_noniid_splits(df, n_clients=N_CLIENTS, alpha=DIRICHLET_ALPHA)

# Summary table
stats = compute_client_statistics(client_dfs)
print('Per-client statistics:')
print(stats[['n_samples', 'positive_rate', 'negative_rate']].to_string())

## 4. Visualise Data Heterogeneity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sample sizes
sizes = [len(client_dfs[i]) for i in range(N_CLIENTS)]
colors = sns.color_palette('Set2')[:N_CLIENTS]
axes[0].bar([f'Client {i}' for i in range(N_CLIENTS)], sizes, color=colors)
axes[0].set_title('Samples per Client')
axes[0].set_ylabel('N Samples')

# Positive rates
rates = [client_dfs[i][TARGET_COLUMN].mean() for i in range(N_CLIENTS)]
axes[1].bar([f'Client {i}' for i in range(N_CLIENTS)], rates, color=colors)
axes[1].axhline(df[TARGET_COLUMN].mean(), linestyle='--', color='black', label=f'Global: {df[TARGET_COLUMN].mean():.3f}')
axes[1].set_title(f'Positive Rate per Client (α={DIRICHLET_ALPHA})')
axes[1].set_ylabel('Positive Rate')
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.suptitle('Client Data Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/client_data_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature-level comparison across clients
key_features = ['BMI', 'Age', 'GenHlth', 'PhysHlth']

fig, axes = plt.subplots(1, len(key_features), figsize=(16, 4))
for ax, feat in zip(axes, key_features):
    data = [client_dfs[i][feat].values for i in range(N_CLIENTS)]
    ax.violinplot(data, positions=range(N_CLIENTS), showmedians=True)
    ax.set_xticks(range(N_CLIENTS))
    ax.set_xticklabels([f'C{i}' for i in range(N_CLIENTS)])
    ax.set_title(feat)

plt.suptitle('Feature Distributions per Client', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/feature_distributions_per_client.png', bbox_inches='tight')
plt.show()

## 5. Save Client Data

In [ ]:
save_client_data(client_dfs)
print('Client data saved to data/federated/')
for i, cdf in client_dfs.items():
    print(f'  client_{i}_data.csv — {len(cdf):,} rows')